In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = "/Users/ann/Library/Mobile Documents/com~apple~CloudDocs/Downloads/归档/linkedin_jobs_cleaned.sqlite"
conn = sqlite3.connect(db_path)

tables_df = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';",
    conn
)
tables = tables_df['name'].tolist()

dfs = {t: pd.read_sql_query(f'SELECT * FROM "{t}";', conn) for t in tables}

df = dfs[tables[0]] if tables else pd.DataFrame()
print(df.head())

conn.close()

df.head()

      job_id  company_id                                              title  \
0     921716   2774458.0                              Marketing Coordinator   
1   10998357  64896719.0                        Assitant Restaurant Manager   
2   23221523    766262.0  Senior Elder Law / Trusts and Estates Associat...   
3   91700727   1481176.0           Economic Development and Planning Intern   
4  103254301  81942316.0                                           Producer   

                                         description  \
0  Job descriptionA leading real estate firm in N...   
1  The National Exemplar is accepting application...   
2  Senior Associate Attorney - Elder Law / Trusts...   
3  Job summary:The Economic Development & Plannin...   
4  Company DescriptionRaw Cereal is a creative de...   

                                         clean_title  
0                              marketing coordinator  
1                        assitant restaurant manager  
2  senior elder law / t

,job_id,company_id,title,description,clean_title
0,921716,2774458.0,Marketing Coordinator,Job descriptionA leading real estate firm in N...,marketing coordinator
1,10998357,64896719.0,Assitant Restaurant Manager,The National Exemplar is accepting application...,assitant restaurant manager
2,23221523,766262.0,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,senior elder law / trusts and estates associat...
3,91700727,1481176.0,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,economic development and planning intern
4,103254301,81942316.0,Producer,Company DescriptionRaw Cereal is a creative de...,producer


In [7]:
import subprocess
import sys

# Install required packages if not already installed
packages = ["sentence-transformers", "torch", "openai", "python-dotenv"]
for package in packages:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All packages installed successfully!")


✅ All packages installed successfully!


In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sentences = df['description'].head(50).tolist()

embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

(50, 384)


In [ ]:
similarities = model.similarity(embeddings, embeddings)
print(similarities)

tensor([[1.0000, 0.3022, 0.4368,  ..., 0.1446, 0.4347, 0.3661],
        [0.3022, 1.0000, 0.2325,  ..., 0.2341, 0.1854, 0.3378],
        [0.4368, 0.2325, 1.0000,  ..., 0.1287, 0.3427, 0.3123],
        ...,
        [0.1446, 0.2341, 0.1287,  ..., 1.0000, 0.1075, 0.3565],
        [0.4347, 0.1854, 0.3427,  ..., 0.1075, 1.0000, 0.3239],
        [0.3661, 0.3378, 0.3123,  ..., 0.3565, 0.3239, 1.0000]])


In [ ]:
import torch

k = 2
idx = torch.randperm(len(sentences))[:k]          
rows = similarities[idx].clone()
rows[torch.arange(k), idx] = -1e9               

j = rows.argmax(dim=1)                           
s = rows[torch.arange(k), j]                   

for i, jj, ss in zip(idx.tolist(), j.tolist(), s.tolist()):
    print(f"{ss:.4f}\nA: {sentences[i]}\nB: {sentences[jj]}\n")


0.5016
A: Glastender Inc. is a family-owned manufacturer of commercial bar and restaurant equipment, known for its high-quality products and innovative solutions. With a strong commitment to the customer experience, Glastender has been serving the industry for over 50 years, providing establishments with state-of-the-art equipment and exceptional service.We are currently looking for an Inside Customer Service Associate who can communicate with outside customers by providing exceptional customer service by addressing their concerns and resolving issues promptly (inquiries, orders, and product information via phone and email). Qualified candidates would be able to perform and possess the following skills:Design bar equipment layouts using the best application of Glastender products.Compile and submit quotations, perform order verification, order entry, and complete detailed shop drawings for use in production.Strong communication and organizational skills and demonstrated attention to de

In [16]:
# LLM Enhancement: Question Answering on Retrieved Results
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

print(f"API Key loaded: {api_key is not None}")
if api_key and api_key.startswith("sk-"):
    print(f"✅ Valid API key format detected")
else:
    print(f"⚠️ API key format invalid or missing")

if not api_key or api_key == "sk-your-new-api-key-here":
    raise ValueError(
        "OPENAI_API_KEY not set correctly. Please:\n"
        "1. Visit https://platform.openai.com/account/api-keys\n"
        "2. Create a new API key\n"
        "3. Update .env file with: OPENAI_API_KEY=sk-your-actual-key\n"
        "4. Do NOT commit .env to Git"
    )

client = OpenAI(api_key=api_key)

def answer_question(query, k=5):
    """
    RAG-based question answering:
    1. Embed the user query
    2. Find top-K similar job descriptions
    3. Use LLM to answer the question based on retrieved results
    """
    try:
        # Encode the query
        query_embedding = model.encode(query)
        
        # Compute similarities with all descriptions in the dataset
        corpus_embeddings = model.encode(df['description'].tolist())
        similarities = model.similarity([query_embedding], corpus_embeddings)[0]
        
        # Get top-K similar documents
        top_indices = np.argsort(-similarities)[:k]
        retrieved_docs = df.iloc[top_indices][['title', 'description']].values.tolist()
        
        # Prepare context from retrieved documents
        context = "\n\n".join(
            [f"Job {i+1}: {title}\nDescription: {desc[:500]}..." 
             for i, (title, desc) in enumerate(retrieved_docs)]
        )
        
        # Create prompt for LLM
        system_prompt = """You are a helpful job market analyst. 
        Based on the job listings provided, answer the user's question accurately and concisely.
        If the answer is not in the provided documents, say so."""
        
        user_prompt = f"""Based on these job listings:

{context}

Please answer this question: {query}"""
        
        # Call OpenAI API (using gpt-3.5-turbo for broader access)
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )
        
        return {
            "query": query,
            "answer": response.choices[0].message.content,
            "retrieved_jobs": [title for title, _ in retrieved_docs]
        }
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise


API Key loaded: True
✅ Valid API key format detected


In [18]:
# Example usage: Ask a question about the job market
result = answer_question("What are common skills required for data science positions?", k=5)
print("Query:", result["query"])
print("\nAnswer:", result["answer"])
print("\nRetrieved job titles:")
for i, title in enumerate(result["retrieved_jobs"], 1):
    print(f"  {i}. {title}")


/opt/anaconda3/lib/python3.13/site-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  a = torch.tensor(a)


Query: What are common skills required for data science positions?

Answer: Common skills required for data science positions based on the job listings provided include:
- Proficiency in programming languages such as SQL, Python, and/or R
- Experience with big data tools such as Hadoop
- Data visualization skills using tools like Tableau
- Strong analytical skills
- Ability to work with large datasets
- Data modeling skills
- Strong communication skills
- Experience in data analytics
- Knowledge of data mining and segmentation techniques
- Proficiency in database design and reporting packages.

Retrieved job titles:
  1. Senior Data Analyst
  2. Senior Data Scientist
  3. Data Analyst 3
  4. Data Specialist
  5. Data Engineer(Informatica / Snaplogic/ SQL)
